# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## 1. Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
scale = 4
init_dask_cluster_eopf(scale=scale)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/3c1acf4794434940914867b8a47857c9/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [3]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

13:31:31.411 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

13:31:31.413 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

13:31:31.414 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

13:31:31.415 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

13:31:31.416 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

13:31:31.417 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

13:31:31.445 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0/config' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [5]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

13:31:34.867 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id '681fd824-e603-46e9-8f60-2b7715bf4306'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/681fd824-e603-46e9-8f60-2b7715bf4306


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [7]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


## 3. Run Prefect flow for S1 short data (~1 minute)

In [8]:
output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'


In [9]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'prehistoric-serval'.
└── UUID: 980091f4-5899-468e-8c6f-43268bca0dc9
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-03-10 13:31:43 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/980091f4-5899-468e-8c6f-43268bca0dc9
Watching flow run 'prehistoric-serval'...


13:31:43.213 | INFO    | prefect - Flow run is in state 'Scheduled'
13:31:48.231 | INFO    | prefect - Flow run is in state 'Pending'
13:31:53.244 | INFO    | prefect - Flow run is in state 'Pending'
13:31:58.260 | INFO    | prefect - Flow run is in state 'Running'
13:32:03.277 | INFO    | prefect - Flow run is in state 'Running'
13:32:08.292 | INFO    | prefect - Flow run is in state 'Running'
13:32:13.307 | INFO    | prefect - Flow run is in state 'Running'
13:32:18.327 | INFO    | prefect - Flow run is in state 'Running'
13:32:23.344 | INFO    | prefect - Flow run is in state 'Running'
13:32:28.358 | INFO    | prefect - Flow run is in state 'Running'
13:32:33.375 | INFO    | prefect - Flow run is in state 'Running'
13:32:38.398 | INFO    | prefect - Flow run is in state 'Running'
13:32:43.418 | INFO    | prefect - Flow run is in state 'Running'
13:32:48.434 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [10]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

Output products generated on: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'
Download reports locally: './l0/reports/s1.short'


## 4. Run Prefect flow for S1 full data (~30 minutes)

In [ ]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")
    
    local_report_dir = osp.join("./l0", "reports", "s1")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [ ]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")

    local_report_dir = osp.join("./l0", "reports", "s3")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway.scale_cluster(dask_cluster.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [11]:
from importlib import reload
debug_flow = False

In [14]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=4)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

Shutting down cluster 'ed7fe039898d4bf494d81a29e94e5c06' ...
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/37d57281c85b454ca770b3574fdb36fc/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [18]:
if debug_flow:
    import first_l0_processor
    reload(first_l0_processor)
    results = first_l0_processor.first_l0_processor(**s1_short)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


13:50:08.002 | INFO    | prefect.engine - Created flow run 'dynamic-bird' for flow 'first-l0-processor'

13:50:08.003 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/7a8d5e64-8578-49ff-a001-7384d9596b65

13:50:08.092 | INFO    | Task run 'dummy_auxip_search-2ef' - Start (dummy) auxip search

13:50:08.095 | INFO    | Task run 'dummy_cadip_search-f39' - Start (dummy) cadip search

...

13:50:51.199 | INFO    | Task run 'dummy_catalog_save-e5a' - Start catalog saving

13:50:52.210 | INFO    | Task run 'dummy_catalog_save-e5a' - End (dummy) catalog saving:

13:50:52.213 | INFO    | Task run 'dummy_catalog_save-e5a' - Finished in state Completed()

13:50:52.243 | INFO    | Flow run 'dynamic-bird' - Finished in state Completed()

{}